In [1]:
import warnings
warnings.filterwarnings('ignore')

# LangChain의 LCEL(LangChain Expression Language)

LangChain의 컴포넌트를 조합하여 복잡한 작업 흐름을 쉽게 구성할 수 있게 해준다.

# 기본 설정

## .env 환경 변수

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

## 기본 라이브러리

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

##  벡터 저장소 로드

In [4]:
# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# 벡터 저장소 설정
# Chroma() 클래스로 로컬에 저장할 때 사용한 임베딩 모델, 테이블 이름, 테이블이 저장된 경로를 넘겨서 벡터 저장소의 문서들을 얻어온다.
vectorstore = Chroma(
    embedding_function=embeddings, # 로컬에 저장할 때 텍스트를 벡터로 변환할 때 사용한 OpenAIEmbeddings로 만든 임베딩 모델을 지정한다.
    collection_name='chroma_test', # 읽어올 Chroma 데이터베이스 내부의 테이블(벡터 저장소) 이름을 지정한다.
    persist_directory='./chroma_db', # 벡터 저장소가 물리적인 파일로 저장된 디렉토리(폴더)를 지정한다.
)

print(f'벡터 저장소에 저장된 문서 개수: {vectorstore._collection.count()}')

벡터 저장소에 저장된 문서 개수: 5


# Prompt와 LLM 연결하기

LCEL을 사용해서 프롬프트와 LLM 연결

<img src="./lcel1.png" width="1200" align="left" />

동적 프롬프트 템플릿 생성 및 활용

<img src="./lcel2.png" width="600" align="left" />

LangChain을 사용해서 LLM(대화형 AI)에게 전달할 프롬프트를 설계한다.  
단순히 질문만 던지는 것이 아니라, AI의 역할(Persona, 정체성)과 사용자의 질문 형식을 미리 정의하는 틀을 만든다.

In [5]:
# 언어 모델을 만든다.
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0, max_completion_tokens=150)

# 프롬프트를 대화 메시지를 큰 리스트에 역할과 메시지를 튜플 형태로 정의한다.
# 역할은 'human', 'user', 'ai', 'assistant', 'system' 중 1개를 사용해야 한다.
messages = [
    # 시스템 메시지라고 하며, AI에게 '너는 어떤 존재인가?'라는 역할을 부여한다.
    ('system', '당신은 유능한 나만의 인공지능 비서입니다.'), # SystemMessagePromptTemplate
    # 사용자 메시지라고 하며, {query}는 나중에 실제 질문 내용으로 치환될 변수(placeholder) 자리를 의미한다.
    ('human', '{query}'), # HumanMessagePromptTemplate
]

# 프롬프트 템플릿을 생성한다.
# 위에서 정의한 튜플이 저장된 리스트 형태의 메시지 구조를 바탕으로, LangChain이 인식할 수 있는 템플릿 객체로 변환한다.
# 이 프롬프트 객체에 질문만 넣어주면 모델이 이해할 수 있는 복잡한 메시지 구조를 만들어 준다.
# from_messages() 메소드에 프롬프트로 구성할 내용을 넘겨서 프롬프트를 만든다.
prompt = ChatPromptTemplate.from_messages(messages)
prompt

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 유능한 나만의 인공지능 비서입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

프롬프트 내부의 변수 목록 확인하기

In [6]:
print(prompt.input_variables)

['query']


프롬프트 내부의 변수에 실제 내용을 채워넣어, 모델에 전달할 최종 메시지 형태 만들기, 변수에 실제 내용을 넣어서 프롬프트 완성하기

In [7]:
# format() 메소드의 인수로 '변수이름=메시지' 형태로 실제 내용을 넣어준다.
prompt_text = prompt.format(query='테슬라 창업자는 누구인가요?')
print(prompt_text)

System: 당신은 유능한 나만의 인공지능 비서입니다.
Human: 테슬라 창업자는 누구인가요?


프롬프트를 AI 모델에게 실제로 보내서 답변을 받아내는 RAG 프로세스를 실행한다.

In [8]:
# LLM 호출 및 답변을 생성한다.
# LCEL 문법을 사용하지(파이프라인을 연결하지) 않고 RAG를 실행할 때는 프롬프트에 format() 메소드로 변수에 실제 내용을 채워넣고 전달한다.
response = llm.invoke(prompt_text)

# RAG 프로세스의 실행 결과는 AIMessage 객체이고 여기에는 답변 내용뿐만 아니라 사용된 토큰 수, 수행 시간 등 다양한 메타 데이터가 포함되어 있다.
response

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 이후 엘론 머스크가 2004년에 투자자로 참여하면서 CEO로 취임하게 되었고, 회사의 비전과 방향성을 주도하게 되었습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 36, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_53c3b1e564', 'id': 'chatcmpl-EJwGBvs9ZjNWC7i34ZFlk0WG1kiaB', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06631-1ffc-79f2-8a77-be7909f7a249-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_t

RAG 프로세스 실행 결과에서 답변 내용만 추출한다.

In [9]:
print(response.content)

테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 이후 엘론 머스크가 2004년에 투자자로 참여하면서 CEO로 취임하게 되었고, 회사의 비전과 방향성을 주도하게 되었습니다.


## LangChain의 LCEL을 사용해서 프롬프트와 모델 연결해서 chain 만들기

`|(파이프 연산자)`는 `|` 왼쪽의 출력을 `|` 오른쪽의 입력으로 보낸다.

In [10]:
# 체인을 생성한다. 파이프라인을 연결한다.
# prompt가 사용자의 입력을 받아 메시지 형태로 가공한다. => 가공된 메시지를 모델에게 전달한다.
# chain에 prompt와 llm이 합쳐진 새로운 객체가 생성된다. 이전 처럼 프롬프트를 만들고 모델을 따로 호출할 필요 없이 chain만 실행하면 답변까지 한 번에 나온다.
chain = prompt | llm
chain

ChatPromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 유능한 나만의 인공지능 비서입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain-openai': '1.6.0'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature

체인의 input_schema 속성은 LangChain의 모든 컴포넌트(Runnable)의 입력 형식을 정의한 스키마(데이터 구조) 객체를 얻어온다.  
스키마에서 schema() 메소드를 실행하면 이 객체를 JSON 형식의 딕셔너리로 변환한다.  

In [11]:
print(chain.input_schema.schema())

{'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'PromptInput', 'type': 'object'}


파이썬의 데이터를 `보기 좋게 출력(Pretty Print)`하기 위해 pprint를 import 한다.

In [12]:
from pprint import pprint

일반 print() 함수는 딕셔너리나 리스트 등 복잡한 구성을 가지는 데이터를 출력할 때 한 줄로 길게 늘어뜨려 읽기 힘들게 보여주지만, pprint() 함수는 이를 줄 바꿈과 들여쓰기를 적용해 깔끔하게 정돈해서 출력한다.

In [13]:
pprint(chain.input_schema.schema())

{'properties': {'query': {'title': 'Query', 'type': 'string'}},
 'required': ['query'],
 'title': 'PromptInput',
 'type': 'object'}


## chain 실행하기

앞에서 `chain = prompt | llm` 형태로 연결해둔 체인을 실제로 동작시켜 질문을 던지고, AI의 최종 답변을 받아온다.

LCEL 문법을 사용하지 않을 경우 format() 메소드로 프롬프트에 실제 내용을 채워넣고 실행하지만 LCEL 문법을 사용하는 경우 프롬프트에 채워넣을 변수 이름을 key로 하고 메시지를 value로 하는 딕셔너리를 넘겨서 실행한다.

In [14]:
# invoke() 메소드의 인수는 {변수 이름: 메시지} 형태의 딕셔너리를 넘겨야 한다.
# invoke() 메소드가 실행되면 인수로 지정된 '테슬라 창업자는 누구인가요?' 메시지가 프롬프트의 {query}라는 변수로 전달되서 프롬프트가 완성되고 완성된 프롬프트 
# 내용이 AI 모델의 입력으로 전달된다.
# AI 모델은 입력받은 프롬프트를 LLM에게 던지고 답변 내용과 메타 데이터가 저장된 AIMessage 객체 형태의 응답을 받는다.
response = chain.invoke({'query': '테슬라 창업자는 누구인가요?'})
response

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 방향성을 크게 변화시켰습니다. 이후 그는 테슬라의 가장 유명한 얼굴이 되었습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 103, 'prompt_tokens': 37, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da56f7d23d', 'id': 'chatcmpl-EJwGCUrinQpFpXX0EGLdgtISFTv50', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06631-256f-72e0-8cea-4bb770798009-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tok

In [15]:
print(response.content)

테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, CEO로 취임하며 회사의 방향성을 크게 변화시켰습니다. 이후 그는 테슬라의 가장 유명한 얼굴이 되었습니다.


In [16]:
# chain이 실행하는 프롬프트는 변수가 1개인 프롬프트는 LangChain이 알아서 {query} 변수에 자동으로 내용을 채워준다.
response = chain.invoke('테슬라 창업자는 누구인가요?')
response

AIMessage(content='테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, 이후 CEO로 취임하여 회사를 이끌어왔습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 37, 'total_tokens': 124, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_da56f7d23d', 'id': 'chatcmpl-EJwGDh853m4LN1F7ZilMdPoEasb9v', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a06631-2a1b-7ab1-9e67-15ed156ad989-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 37, 'output_tokens': 87, 'tot

# Prompt와 LLM과 OutputParser 연결하기

## 문자열 파싱 - StrOutputParser

모델의 응답인 AIMessage 객체에서 순수 응답 결과(content)를 문자열만 얻어오기 위해서 StrOutputParser를 import 한다.

In [17]:
from langchain_core.output_parsers import StrOutputParser

In [18]:
# 문자열 파서 StrOutputParser의 객체를 만든다. AIMessage 객체에서 content 부분만 얻어올 수 있다.
output_parser = StrOutputParser()

# 문자열 파서를 실행한다.
# invoke() 메소드의 인수로 LLM의 응답 결과(AIMessage 객체)를 넘겨셔 문자열만 얻어온다. 인수로 넘긴 내용이 순수한 문자열이면 문자열이 그대로 반환된다.
output_parser.invoke(response)

'테슬라의 창립자는 엘론 머스크(Elon Musk)입니다. 그러나 테슬라는 2003년에 마틴 에버하드(Martin Eberhard)와 마크 타페닝(Mark Tarpenning)에 의해 설립되었습니다. 엘론 머스크는 2004년에 투자자로 참여한 후, 이후 CEO로 취임하여 회사를 이끌어왔습니다.'

문자열 파서를 LCEL 방식으로 연결한다.

In [19]:
# invoke() 메소드가 실행되면 질문이 프롬프로 전달되고 완성된 프롬프트가 LLM으로 전달되고 LLM의 응답 결과 문자열 파서로 전달되서 문자열만 추출한다.
str_chain = prompt | llm | output_parser
response = str_chain.invoke({'query': '리비안의 설립 년도는 언제인가요?'})

In [20]:
response

'리비안(Rivian)은 2009년에 설립되었습니다. 이 회사는 전기차를 개발하고 생산하는 데 주력하고 있습니다.'

## JSON 파싱 - JsonOutputParser

LLM의 응답 결과(AIMessage 객체)에는 JSON 형태의 데이터가 포함되므로 이를 딕셔너리나 리스트 구조로 변환해서 사용하면 편리하다.

데이터를 JSON 형식으로 해석하고 변환하기 위해서 JsonOutputParser를 import 한다.

In [21]:
from langchain_core.output_parsers import JsonOutputParser

In [25]:
# JSON 파서 JsonOutputParser의 객체를 만든다.
json_parser = JsonOutputParser()

# chain은 prompt와 llm이 연결된 파이프라인이다.
# 질문에 'JSON 형식'으로 응답해달라고 요청했으므로 모델은 대략 {"회사명": "테슬라", ...}와 같은 문자열이 반환된다.
# 리턴된 값은 JSON을 가장한 문자열 형태이므로 파이썬에서 바로 다루기 어려운 상태이다.
response = chain.invoke({'query': '테슬라 창업자는 누구인가요? JSON 형식으로 응답해주세요. 모든 내용을 한글로 출력해주세요.'})
print(type(response.content))
print(response)
print('-' * 100)

# JSON 형식으로 파싱한다.
json_response = json_parser.invoke(response)
print(type(json_response))
print(json_response)

<class 'str'>
content='```json\n{\n  "회사": "테슬라",\n  "창업자": [\n    {\n      "이름": "엘론 머스크",\n      "역할": "CEO 및 제품 아키텍트"\n    },\n    {\n      "이름": "마틴 에버하드",\n      "역할": "공동 창립자"\n    },\n    {\n      "이름": "마크 타페닝",\n      "역할": "공동 창립자"\n    },\n    {\n      "이름": "JB 스트라우벨",\n      "역할": "공동 창립자 및 CTO"\n    }\n  ],\n  "설립연도": 2003\n}\n```' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 150, 'prompt_tokens': 53, 'total_tokens': 203, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4f8a068d33', 'id': 'chatcmpl-EJwIjSyIHJj64BqUNcb3BPbrno9Nq', 'service_tier': 'default', 'finish_reason': 'length', 'logpr

In [27]:
pprint(json_response['창업자'])

[{'역할': 'CEO 및 제품 아키텍트', '이름': '엘론 머스크'},
 {'역할': '공동 창립자', '이름': '마틴 에버하드'},
 {'역할': '공동 창립자', '이름': '마크 타페닝'},
 {'역할': '공동 창립자 및 CTO', '이름': 'JB 스트라우벨'}]


# Schema 지정 - PydanticOutputPaeser

LLM의 응답 결과(AIMessage 객체)로 부터 우리가 정의한 데이터 모델(특정 클래스 구조)에 맞는 답변을 받아낸다.

모델의 답변을 pydantic 클래스 객체로 변환하기 위해 PydanticOutputParser를 import 한다.

In [28]:
from langchain_core.output_parsers import PydanticOutputParser

LangChain이 업데이트되면서 내부 패키지 구조가 변경되서 langchain_core.pydantic_v1를 더 이상 지원하지 않는다.  
`from langchain_core.pydantic_v1 import BaseModel, Field, validator`  
현재의 LangChain은 langchain_core.pydantic_v1 대신 파이썬의 표준 pydantic을 직접 사용한다.

`BaseModel`: pydantic에서 모델을 정의할 때 상속받는 최상위 클래스이다.  
`Field`: 모델 내부 각 필드의 세부 설정 및 제약 조건을 정의한다.  
`validator`: 기본 타입 검사 외에 사용자가 직접 커스텀 검증 로직을 추가할 때 사용하는 데코레이터이다.

In [30]:
from pydantic import BaseModel, Field, validator

데이터 모델을 정의한다.  
인물 정보를 담을 Person이라는 클래스를 pydantic의 BaseModel 클래스를 상속받아 만든다.

In [46]:
# BaseModel 클래스를 상속받으면 이 클래스는 단순한 클래스가 아니라, 데이터 타입 검증과 자동 형변환 기능이 내장된 pydantic 데이터 모델로 동작한다.
class Person(BaseModel):
    # 독스트링(docstring)으로 클래스의 설명이다.
    # AI 프레임워크와 연결될 때 AI 모델에게 이 클래스가 전체적으로 무엇을 의미하는지 안내하는 힌트(prompt)역할을 한다.
    '''사람과 그 사람의 직함 또는 직위에 대한 정보.'''
    # 'name: str'는 name 변수를 정의하며, 저장되는 데이터는 반드시 문자열(str)이어야 함을 명시한다.
    # '...'은 이 값이 필수 입력 항목임을 뜻한다. 데이터가 입력될 때 name 값이 누락되면 에러가 발생된다.
    # description 속성은 필드에 대한 설명 메타 데이터로 AI가 데이터를 분석해서 JSON 데이터로 변환할 때, 어떤 정보를 할당할지 판단하는 가이드라인으로 활용된다.
    name: str = Field(..., description='사람의 이름')
    # 'title: str'는 title 변수를 정의하며, 저장되는 데이터는 반드시 문자열(str)이어야 함을 명시한다.
    title: str = Field(..., description='사람의 직함 또는 직위')

In [47]:
# 위에서 만든 Person 클래스를 기준으로 작동하는 파서를 만든다.
# 이 파서는 AI의 답변을 감시하며 '이름'과 '직함 또는 직위'가 제대로 들어왔는 확인하고 파이썬 객체로 바꿔줄 준비를 한다.
person_parser = PydanticOutputParser(pydantic_object=Person)

print('PydanticOutputParser 프롬프트')
# get_format_instructions() 메소드는 AI가 답변을 어떻게 JSON 형식으로 구성해야 하는지 설명하는 프롬프트를 자동으로 생성한다.
print(person_parser.get_format_instructions())

PydanticOutputParser 프롬프트
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"description": "사람과 그 사람의 직함 또는 직위에 대한 정보.", "properties": {"name": {"description": "사람의 이름", "title": "Name", "type": "string"}, "title": {"description": "사람의 직함 또는 직위", "title": "Title", "type": "string"}}, "required": ["name", "title"]}
```
